In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

BRONZE_SCHEMA = f"{CATALOG}.{SCHEMA_PREFIX}_bronze"
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA_PREFIX}/raw_data"

In [0]:
import logging

logger = logging.getLogger("lab4.bronze_ingestion")
logger.setLevel(logging.INFO)
logger.propagate = False
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s - %(message)s"))
    logger.addHandler(handler)

In [0]:
def copy_into_bronze(table_name, file_pattern):
    result = spark.sql(f"""
        COPY INTO {BRONZE_SCHEMA}.{table_name}
        FROM (
            SELECT
                *,
                _metadata.file_name AS source_filename,
                current_timestamp() AS ingestion_timestamp,
                current_date()      AS load_date
            FROM '{RAW_PATH}'
        )
        FILEFORMAT = CSV
        PATTERN = '{file_pattern}'
        FORMAT_OPTIONS (
            'header' = 'true',
            'inferSchema' = 'true',
            'mergeSchema' = 'true',
            'rescuedDataColumn' = '_rescued_data'
        )
        COPY_OPTIONS ('mergeSchema' = 'true')
    """).first()
    logger.info("%s: %d rows inserted", table_name, result["num_inserted_rows"])

In [0]:
copy_into_bronze("customers", "customers*.csv")
copy_into_bronze("products", "products*.csv")